In [1]:
%spark.pyspark

spark


UsageError: Line magic function `%spark.pyspark` not found.


In [ ]:
%spark.pyspark

!ls /home/data/raw


In [ ]:
%spark.pyspark

import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from itertools import product
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.regression import GBTRegressor, GeneralizedLinearRegression, RandomForestRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import Row
from pyspark.sql import functions
from pyspark.sql.functions import col, mean, month, round, sum, to_date, when
from pyspark.sql.window import Window

from pyspark.sql.functions import max as pyspark_sql_functions_max
from pyspark.sql.functions import min as pyspark_sql_functions_min

pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', None)


In [ ]:
%spark.pyspark

location = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/home/data/raw/locationData.csv")

weather = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/home/data/raw/weatherData.csv")
    

In [ ]:
%spark.pyspark

location.printSchema()


In [ ]:
%spark.pyspark

weather.printSchema()


In [ ]:
%spark.pyspark

raw_data = weather.join(location, "location_id")


In [ ]:
%spark.pyspark

null_counts = raw_data.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in raw_data.columns
])

null_counts.toPandas()


In [ ]:
%spark.pyspark

data_date_parsed = raw_data.withColumn("date_parsed", to_date(col("date"), "M/d/yyyy"))


In [ ]:
%spark.pyspark

data_filtered_for_may = data_date_parsed.filter(month(col("date_parsed")) == 5)


In [ ]:
%spark.pyspark

data_filtered_for_selected = data_filtered_for_may.select(
    col("precipitation_hours (h)").alias("precipitation_hours"),
    col("sunshine_duration (s)").alias("sunshine_duration"),
    col("wind_speed_10m_max (km/h)").alias("wind_speed"),
    col("et0_fao_evapotranspiration (mm)").alias("evapotranspiration")
)


In [ ]:
%spark.pyspark

data_filtered_for_selected.show()


In [ ]:
%spark.pyspark

data_filtered_for_selected.count()


In [ ]:
%spark.pyspark

data_dropped_duplicate_rows = data_filtered_for_selected.dropDuplicates()


In [ ]:
%spark.pyspark

data_dropped_duplicate_rows.count()


In [ ]:
%spark.pyspark

data_selected_unique_features = (
    data_dropped_duplicate_rows
    .groupBy(
        "precipitation_hours",
        "sunshine_duration",
        "wind_speed"
    )
    .agg(
        mean("evapotranspiration").alias("evapotranspiration")
    )
)


In [ ]:
%spark.pyspark

data_selected_unique_features.count()


In [ ]:
%spark.pyspark

assembler = VectorAssembler(
    inputCols=["precipitation_hours", "sunshine_duration", "wind_speed"],
    outputCol="features"
)

data_vector_assembled = assembler.transform(data_selected_unique_features)


In [ ]:
%spark.pyspark

data_vector_assembled.show()


In [ ]:
%spark.pyspark

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)

scaler_model = scaler.fit(data_vector_assembled)

data_scaled = scaler_model.transform(data_vector_assembled)


In [ ]:
%spark.pyspark

data_scaled.show(truncate=False)


In [ ]:
%spark.pyspark

processed_data = data_scaled.select(col("scaled_features").alias("features"), col("evapotranspiration").alias("label"))


In [ ]:
%spark.pyspark

print("Total processed data rows:", processed_data.count())


In [ ]:
%spark.pyspark

num_bins = 1200

processed_data_binned = processed_data.withColumn(
    "label_bin",
    functions.ntile(num_bins).over(Window.orderBy("label"))
)

fractions = (processed_data_binned
    .groupBy("label_bin")
    .count()
    .select("label_bin", functions.lit(0.8).alias("fraction"))
    .rdd
    .map(lambda row: (row["label_bin"], row["fraction"]))
    .collectAsMap()
)

train_data_df = processed_data_binned.sampleBy("label_bin", fractions, seed=42)
test_data_df = processed_data_binned.subtract(train_data_df)

train_data = train_data_df.drop("label_bin")
test_data = test_data_df.drop("label_bin")

In [ ]:
%spark.pyspark

train_labels = train_data.select("label").toPandas()
test_labels = test_data.select("label").toPandas()

# Plot histograms of train and test
plt.figure(figsize=(10,6))
plt.hist(train_labels['label'], bins=50, alpha=0.6, label='Train', color='blue', density=True)
plt.hist(test_labels['label'], bins=50, alpha=0.6, label='Test', color='orange', density=True)
plt.xlabel("Label")
plt.ylabel("Density")
plt.title("Target Distribution: Train vs Test")
plt.legend()
plt.show()

In [ ]:
%spark.pyspark

print("Total rows in Train:", train_data.count())
print("Total rows in Test:", test_data.count())


In [ ]:
%spark.pyspark

print("\nTrain statistics:")
train_data.select("label").describe().show()

print("\nTest statistics:")
test_data.select("label").describe().show()

In [ ]:
%spark.pyspark

print("\nTrain counts per bin:")
train_data_df.groupBy("label_bin").count().orderBy("label_bin").show(n=num_bins)


In [ ]:
%spark.pyspark

print("\nTest counts per bin:")
test_data_df.groupBy("label_bin").count().orderBy("label_bin").show(n=num_bins)


In [ ]:
%spark.pyspark

train_data.cache()
test_data.cache()


In [ ]:
%spark.pyspark

evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)


In [ ]:
%spark.pyspark

glr_gaussian = GeneralizedLinearRegression(
    featuresCol="features",
    labelCol="label",
    family="gaussian",
    link="log"
)

glr_gaussian_param_grid = (
    ParamGridBuilder()
    .addGrid(glr_gaussian.regParam, [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.05, 0.1, 0.3])
    .build()
)

glr_gaussian_cv = CrossValidator(
    estimator=glr_gaussian,
    estimatorParamMaps=glr_gaussian_param_grid,
    evaluator=evaluator,
    numFolds=5
)

glr_gaussian_model = glr_gaussian_cv.fit(train_data)


In [ ]:
%spark.pyspark

glr_gamma = GeneralizedLinearRegression(
    featuresCol="features",
    labelCol="label",
    family="gamma",
    link="log"
)

glr_gamma_param_grid = (
    ParamGridBuilder()
    .addGrid(glr_gaussian.regParam, [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.05, 0.1, 0.3])
    .build()
)

glr_gamma_cv = CrossValidator(
    estimator=glr_gamma,
    estimatorParamMaps=glr_gamma_param_grid,
    evaluator=evaluator,
    numFolds=5
)

glr_gamma_model = glr_gamma_cv.fit(train_data)


In [ ]:
%spark.pyspark

rfr = RandomForestRegressor(
    featuresCol="features",
    labelCol="label",
    seed=42
)

rfr_param_grid = (
    ParamGridBuilder()
    .addGrid(rfr.numTrees, [50, 75, 100])
    .addGrid(rfr.maxDepth, [5, 10, 15]) 
    .build()
)

rfr_cv = CrossValidator(
    estimator=rfr,
    estimatorParamMaps=rfr_param_grid,
    evaluator=evaluator,
    numFolds=5
)

rfr_model = rfr_cv.fit(train_data)


In [ ]:
%spark.pyspark

gbtr = GBTRegressor(
    featuresCol="features",
    labelCol="label",
    seed=42
)

gbtr_param_grid = (
    ParamGridBuilder()
    .addGrid(gbtr.maxIter, [150, 175, 200])
    .addGrid(gbtr.maxDepth, [1, 3, 5])
    .build()
)

gbtr_cv = CrossValidator(
    estimator=gbtr,
    estimatorParamMaps=gbtr_param_grid,
    evaluator=evaluator,
    numFolds=5
)

gbtr_model = gbtr_cv.fit(train_data)


In [ ]:
%spark.pyspark

glr_gaussian_best_model_estimator = glr_gaussian_model.bestModel
glr_gamma_best_model_estimator = glr_gamma_model.bestModel
rfr_best_model_estimator = rfr_model.bestModel
gbtr_best_model_estimator = gbtr_model.bestModel

glr_gaussian_best_model_param_map = glr_gaussian_best_model_estimator.extractParamMap()
glr_gamma_best_model_param_map = glr_gamma_best_model_estimator.extractParamMap()
rfr_best_model_param_map = rfr_best_model_estimator.extractParamMap()
gbtr_best_model_param_map = gbtr_best_model_estimator.extractParamMap()


In [ ]:
%spark.pyspark


def get_model_cv_params_df(
    cv_model,
    model_name,
    param_grid
):
    best_estimator = cv_model.bestModel

    best_param_map = {
        p.name: v for p, v in best_estimator.extractParamMap().items()
    }

    cv_params = {}
    for grid in param_grid:
        for p, v in grid.items():
            cv_params.setdefault(p.name, set()).add(v)

    cv_params = {
        k: sorted(list(v)) for k, v in cv_params.items()
    }

    rows = []
    for param in sorted(set(cv_params) | set(best_param_map)):
        rows.append({
            "model": model_name,
            "parameter": param,
            "cv_values": cv_params.get(param, None),
            "selected_value": best_param_map.get(param, None)
        })

    return pd.DataFrame(rows)


In [ ]:
%spark.pyspark

glr_gaussian_params_df = get_model_cv_params_df(
    cv_model=glr_gaussian_model,
    model_name="GeneralizedLinearRegression (gaussian)",
    param_grid=glr_gaussian_param_grid
)

glr_gamma_params_df = get_model_cv_params_df(
    cv_model=glr_gamma_model,
    model_name="GeneralizedLinearRegression (gamma)",
    param_grid=glr_gamma_param_grid
)

rfr_params_df = get_model_cv_params_df(
    cv_model=rfr_model,
    model_name="RandomForestRegressor",
    param_grid=rfr_param_grid
)

gbtr_params_df = get_model_cv_params_df(
    cv_model=gbtr_model,
    model_name="GBTRegressor",
    param_grid=gbtr_param_grid
)

all_params_df = pd.concat([
    glr_gaussian_params_df,
    glr_gamma_params_df,
    rfr_params_df,
    gbtr_params_df
], ignore_index=True)


In [ ]:
%spark.pyspark

all_params_df[all_params_df['cv_values'].notna()]


In [ ]:
%spark.pyspark

glr_gaussian_model_predictions_train_data = glr_gaussian_model.transform(train_data)
glr_gaussian_model_predictions_test_data = glr_gaussian_model.transform(test_data)

glr_gamma_model_predictions_train_data = glr_gamma_model.transform(train_data)
glr_gamma_model_predictions_test_data = glr_gamma_model.transform(test_data)

rfr_model_predictions_train_data = rfr_model.transform(train_data)
rfr_model_predictions_test_data = rfr_model.transform(test_data)

gbtr_model_predictions_train_data = gbtr_model.transform(train_data)
gbtr_model_predictions_test_data = gbtr_model.transform(test_data)


In [ ]:
%spark.pyspark

evaluator_rmse = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_mae = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="mae"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="r2"
)

def evaluate_model(predictions, model_name, data_type):
    return Row(
        Model=model_name,
        Data=data_type,
        RMSE=evaluator_rmse.evaluate(predictions),
        MAE=evaluator_mae.evaluate(predictions),
        R2=evaluator_r2.evaluate(predictions)
    )
    

In [ ]:
%spark.pyspark

results = []

results.append(evaluate_model(glr_gaussian_model_predictions_train_data, "GeneralizedLinearRegression (gaussian)", "Train"))
results.append(evaluate_model(glr_gaussian_model_predictions_test_data, "GeneralizedLinearRegression (gaussian)", "Test"))
results.append(evaluate_model(glr_gamma_model_predictions_train_data, "GeneralizedLinearRegression (gamma)", "Train"))
results.append(evaluate_model(glr_gamma_model_predictions_test_data, "GeneralizedLinearRegression (gamma)", "Test"))
results.append(evaluate_model(rfr_model_predictions_train_data, "RandomForestRegressor", "Train"))
results.append(evaluate_model(rfr_model_predictions_test_data, "RandomForestRegressor", "Test"))
results.append(evaluate_model(gbtr_model_predictions_train_data, "GBTRegressor", "Train"))
results.append(evaluate_model(gbtr_model_predictions_test_data, "GBTRegressor", "Test"))

In [ ]:
%spark.pyspark

results_df = spark.createDataFrame(results)
results_df_pd  = (results_df.select("Model", "Data", "MAE", "RMSE", "R2").toPandas())
results_df.select("Model", "Data", "RMSE", "MAE", "R2").show(truncate=False)


In [ ]:
%spark.pyspark

metrics_pd = (
    results_df
    .select("Model", "Data", "MAE", "RMSE", "R2")
    .toPandas()
)

metrics_pd["Data"] = metrics_pd["Data"].replace({
    "train_data": "Train",
    "test_data": "Test"
})


def plot_metric_bar(metric_name):
    fig, ax = plt.subplots(figsize=(10, 6))

    models = metrics_pd["Model"].unique()
    x = np.arange(len(models))
    width = 0.35

    train_vals = (
        metrics_pd[metrics_pd["Data"] == "Train"]
        .set_index("Model")[metric_name]
        .loc[models]
    )

    test_vals = (
        metrics_pd[metrics_pd["Data"] == "Test"]
        .set_index("Model")[metric_name]
        .loc[models]
    )

    ax.bar(x - width/2, train_vals, width, label="Train")
    ax.bar(x + width/2, test_vals, width, label="Test")

    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=30, ha="right")
    ax.set_ylabel(metric_name)
    ax.set_title(f"{metric_name} Comparison Across Best Models")
    ax.legend()
    ax.grid(axis="y")

    plt.tight_layout()
    plt.show()


In [ ]:
%spark.pyspark
%matplotlib inline

plot_metric_bar("RMSE")


In [ ]:
%spark.pyspark
%matplotlib inline

plot_metric_bar("MAE")


In [ ]:
%spark.pyspark
%matplotlib inline

plot_metric_bar("R2")


In [ ]:
%spark.pyspark

def to_pandas(df, label_col="label", pred_col="prediction", sample_frac=1.0):
    sdf = df.select(label_col, pred_col)
    if sample_frac < 1.0:
        sdf = sdf.sample(fraction=sample_frac, seed=42)
    return sdf.toPandas()


def plot_predicted_vs_actual(train_df, test_df, model_name):
    train_pd = to_pandas(train_df)
    test_pd  = to_pandas(test_df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for df, title, ax in [
        (train_pd, "Train Data", axes[0]),
        (test_pd,  "Test Data",  axes[1])
    ]:
        ax.scatter(df["label"], df["prediction"], alpha=0.5)

        min_val = min([df["label"].min(), df["prediction"].min()])
        max_val = max([df["label"].max(), df["prediction"].max()])
        ax.plot([min_val, max_val], [min_val, max_val], linestyle="--")

        ax.set_title(title)
        ax.set_xlabel("Actual")
        ax.set_ylabel("Predicted")
        ax.grid(True)

    fig.suptitle(f"Predicted vs Actual – {model_name}", fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
%spark.pyspark
%matplotlib inline

plot_predicted_vs_actual(
    glr_gaussian_model_predictions_train_data,
    glr_gaussian_model_predictions_test_data,
    "Generalized Linear Regression (Gaussian)"
)


In [ ]:
%spark.pyspark
%matplotlib inline

plot_predicted_vs_actual(
    glr_gamma_model_predictions_train_data,
    glr_gamma_model_predictions_test_data,
    "Generalized Linear Regression (Gamma)"
)


In [ ]:
%spark.pyspark
%matplotlib inline

plot_predicted_vs_actual(
    rfr_model_predictions_train_data,
    rfr_model_predictions_test_data,
    "Random Forest Regressor"
)


In [ ]:
%spark.pyspark
%matplotlib inline

plot_predicted_vs_actual(
    gbtr_model_predictions_train_data,
    gbtr_model_predictions_test_data,
    "Gradient-Boosted Trees Regressor"
)


In [ ]:
%spark.pyspark

def get_residual_df(df):
    return (
        df.selectExpr(
            "label as actual",
            "label - prediction as residual"
        )
        .toPandas()
    )


def plot_residuals_vs_actual(train_df, test_df, model_name):
    train_pd = get_residual_df(train_df)
    test_pd  = get_residual_df(test_df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Train
    axes[0].scatter(train_pd["actual"], train_pd["residual"], alpha=0.5)
    axes[0].axhline(0, linestyle="--")
    axes[0].set_title("Train Data")
    axes[0].set_xlabel("Actual")
    axes[0].set_ylabel("Residual (Actual − Predicted)")
    axes[0].grid(True)

    # Test
    axes[1].scatter(test_pd["actual"], test_pd["residual"], alpha=0.5)
    axes[1].axhline(0, linestyle="--")
    axes[1].set_title("Test Data")
    axes[1].set_xlabel("Actual")
    axes[1].set_ylabel("Residual (Actual − Predicted)")
    axes[1].grid(True)

    fig.suptitle(f"Residual Plot – {model_name}", fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
%spark.pyspark
%matplotlib inline

plot_residuals_vs_actual(
    glr_gaussian_model_predictions_train_data,
    glr_gaussian_model_predictions_test_data,
    "Generalized Linear Regression (Gaussian)"
)


In [ ]:
%spark.pyspark
%matplotlib inline

plot_residuals_vs_actual(
    glr_gamma_model_predictions_train_data,
    glr_gamma_model_predictions_test_data,
    "Generalized Linear Regression (Gamma)"
)


In [ ]:
%spark.pyspark
%matplotlib inline

plot_residuals_vs_actual(
    rfr_model_predictions_train_data,
    rfr_model_predictions_test_data,
    "Random Forest Regressor"
)


In [ ]:
%spark.pyspark
%matplotlib inline

plot_residuals_vs_actual(
    gbtr_model_predictions_train_data,
    gbtr_model_predictions_test_data,
    "Gradient-Boosted Trees Regressor"
)


In [ ]:
%spark.pyspark

def get_residuals(df):
    return (
        df.selectExpr("label - prediction as residual")
          .toPandas()
    )
    

def plot_residual_distributions(train_df, test_df, model_name, bins=50):
    train_res = get_residuals(train_df)
    test_res  = get_residuals(test_df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Train residuals
    axes[0].hist(train_res["residual"], bins=bins, alpha=0.7)
    axes[0].axvline(0, linestyle="--")
    axes[0].set_title("Train Residuals")
    axes[0].set_xlabel("Residual (Actual − Predicted)")
    axes[0].set_ylabel("Frequency")
    axes[0].grid(True)

    # Test residuals
    axes[1].hist(test_res["residual"], bins=bins, alpha=0.7)
    axes[1].axvline(0, linestyle="--")
    axes[1].set_title("Test Residuals")
    axes[1].set_xlabel("Residual (Actual − Predicted)")
    axes[1].set_ylabel("Frequency")
    axes[1].grid(True)

    fig.suptitle(f"Residual Distributions – {model_name}", fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
%spark.pyspark
%matplotlib inline

plot_residual_distributions(
    glr_gaussian_model_predictions_train_data,
    glr_gaussian_model_predictions_test_data,
    "Generalized Linear Regression (Gaussian)"
)


In [ ]:
%spark.pyspark
%matplotlib inline

plot_residual_distributions(
    glr_gamma_model_predictions_train_data,
    glr_gamma_model_predictions_test_data,
    "Generalized Linear Regression (Gamma)"
)


In [ ]:
%spark.pyspark
%matplotlib inline

plot_residual_distributions(
    rfr_model_predictions_train_data,
    rfr_model_predictions_test_data,
    "Random Forest Regressor"
)


In [ ]:
%spark.pyspark
%matplotlib inline

plot_residual_distributions(
    gbtr_model_predictions_train_data,
    gbtr_model_predictions_test_data,
    "Gradient-Boosted Trees Regressor"
)


In [ ]:
%spark.pyspark

rf_importance = rfr_model.bestModel.featureImportances.toArray()
gbt_importance = gbtr_model.bestModel.featureImportances.toArray()

feature_names = assembler.getInputCols()

rf_imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_importance
}).sort_values("importance", ascending=False)

gbt_imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance": gbt_importance
}).sort_values("importance", ascending=False)


def plot_feature_importance(df, model_name):
    plt.figure(figsize=(10, 6))
    plt.barh(df["feature"], df["importance"])
    plt.gca().invert_yaxis()
    plt.xlabel("Importance Score")
    plt.title(f"Feature Importances – {model_name}")
    plt.grid(axis="x")
    plt.tight_layout()
    plt.show()


In [ ]:
%spark.pyspark
%matplotlib inline

plot_feature_importance(
    rf_imp_df,
    "Random Forest Regressor"
)


In [ ]:
%spark.pyspark
%matplotlib inline

plot_feature_importance(
    gbt_imp_df,
    "Gradient-Boosted Trees Regressor"
)


In [ ]:
%spark.pyspark

data_filtered_for_min_max = data_filtered_for_selected.select(
    pyspark_sql_functions_min("precipitation_hours").alias("min_precipitation_hours"),
    pyspark_sql_functions_max("precipitation_hours").alias("max_precipitation_hours"),
    pyspark_sql_functions_min("sunshine_duration").alias("min_sunshine_duration"),
    pyspark_sql_functions_max("sunshine_duration").alias("max_sunshine_duration"),
    pyspark_sql_functions_min("wind_speed").alias("min_wind_speed"),
    pyspark_sql_functions_max("wind_speed").alias("max_wind_speed")
)

In [ ]:
%spark.pyspark

data_filtered_for_min_max.show(truncate=False)

In [ ]:
%spark.pyspark

min_precipitation_hours = 0
max_precipitation_hours = 24
samples_precipitation_hours = int((max_precipitation_hours - min_precipitation_hours) * 4) + 1

min_sunshine_duration = 0.0
max_sunshine_duration = math.ceil(data_filtered_for_min_max.select("max_sunshine_duration").first()[0] / 3600) * 3600
samples_sunshine_duration = int((max_sunshine_duration - min_sunshine_duration) * 4 / 3600) + 1

min_wind_speed = 0
max_wind_speed = math.ceil(data_filtered_for_min_max.select("max_wind_speed").first()[0] / 10) * 10
samples_wind_speed = int((max_wind_speed - min_wind_speed)) + 1


In [ ]:
%spark.pyspark

precipitation_hours_range = np.linspace(min_precipitation_hours, max_precipitation_hours, samples_precipitation_hours)
sunshine_duration_range = np.linspace(min_sunshine_duration, max_sunshine_duration, samples_sunshine_duration)
wind_speed_range = np.linspace(min_wind_speed, max_wind_speed, samples_wind_speed)


In [ ]:
%spark.pyspark

scenarios = list(product(
    precipitation_hours_range,
    sunshine_duration_range,
    wind_speed_range
))

scenarios = [
    (float(p), float(s), float(w))
    for p, s, w in scenarios
]

scenario_df = spark.createDataFrame(
    scenarios,
    ["precipitation_hours", "sunshine_duration", "wind_speed"]
)


In [ ]:
%spark.pyspark

scenario_df.count()


In [ ]:
%spark.pyspark

scenario_df_assembled = assembler.transform(scenario_df)
scenario_df_scaled = scaler_model.transform(scenario_df_assembled)


In [ ]:
%spark.pyspark

scenario_df_scaled.show(truncate=False)


In [ ]:
%spark.pyspark

scenario_df_scaled_predictions_glr_gaussian_model = glr_gaussian_model.transform(scenario_df_scaled)
scenario_df_scaled_predictions_glr_gamma_model = glr_gamma_model.transform(scenario_df_scaled)
scenario_df_scaled_predictions_rfr_model = rfr_model.transform(scenario_df_scaled)
scenario_df_scaled_predictions_gbtr_model = gbtr_model.transform(scenario_df_scaled)


In [ ]:
%spark.pyspark

scenario_df_scaled_predictions_glr_gaussian_model.show(truncate=False)


In [ ]:
%spark.pyspark

scenario_df_scaled_predictions_glr_gamma_model.show(truncate=False)


In [ ]:
%spark.pyspark

scenario_df_scaled_predictions_rfr_model.show(truncate=False)


In [ ]:
%spark.pyspark

scenario_df_scaled_predictions_gbtr_model.show(truncate=False)


In [ ]:
%spark.pyspark

low_et_predictions_glr_gaussian_model = scenario_df_scaled_predictions_glr_gaussian_model.filter(scenario_df_scaled_predictions_glr_gaussian_model.prediction < 1.5)
low_et_predictions_glr_gamma_model = scenario_df_scaled_predictions_glr_gamma_model.filter(scenario_df_scaled_predictions_glr_gamma_model.prediction < 1.5)
low_et_predictions_rfr_model = scenario_df_scaled_predictions_rfr_model.filter(scenario_df_scaled_predictions_rfr_model.prediction < 1.5)
low_et_predictions_gbtr_model = scenario_df_scaled_predictions_gbtr_model.filter(scenario_df_scaled_predictions_gbtr_model.prediction < 1.5)


In [ ]:
%spark.pyspark

print("Low ET predictions (< 1.5):")

print("GLR Gaussian model:", low_et_predictions_glr_gaussian_model.count())
print("GLR Gamma model   :", low_et_predictions_glr_gamma_model.count())
print("Random Forest     :", low_et_predictions_rfr_model.count())
print("Gradient Boosted  :", low_et_predictions_gbtr_model.count())


In [ ]:
%spark.pyspark

low_et_predictions_glr_gaussian_model.show()


In [ ]:
%spark.pyspark

low_et_predictions_glr_gamma_model.show()


In [ ]:
%spark.pyspark

low_et_predictions_rfr_model.show()


In [ ]:
%spark.pyspark

low_et_predictions_gbtr_model.show()


In [ ]:
%spark.pyspark

mean_values_glr_gaussian_model = low_et_predictions_glr_gaussian_model.select(
    mean('precipitation_hours').alias('mean_precipitation_hours'),
    mean('sunshine_duration').alias('mean_sunshine_duration'),
    mean('wind_speed').alias('mean_wind_speed')
).collect()[0]

print("Predicted mean conditions for May 2026 with evapotranspiration < 1.5 mm: GeneralizedLinearRegression (gaussian) Model")
print(f"Precipitation hours: {mean_values_glr_gaussian_model['mean_precipitation_hours']:.2f}")
print(f"Sunshine duration: {mean_values_glr_gaussian_model['mean_sunshine_duration']:.2f}")
print(f"Wind speed: {mean_values_glr_gaussian_model['mean_wind_speed']:.2f}")

In [ ]:
%spark.pyspark

mean_values_glr_gamma_model = low_et_predictions_glr_gamma_model.select(
    mean('precipitation_hours').alias('mean_precipitation_hours'),
    mean('sunshine_duration').alias('mean_sunshine_duration'),
    mean('wind_speed').alias('mean_wind_speed')
).collect()[0]

print("Predicted mean conditions for May 2026 with evapotranspiration < 1.5 mm: GeneralizedLinearRegression (gamma) Model")
print(f"Precipitation hours: {mean_values_glr_gamma_model['mean_precipitation_hours']:.2f}")
print(f"Sunshine duration: {mean_values_glr_gamma_model['mean_sunshine_duration']:.2f}")
print(f"Wind speed: {mean_values_glr_gamma_model['mean_wind_speed']:.2f}")

In [ ]:
%spark.pyspark

mean_input_glr_gaussian_df = spark.createDataFrame([
    Row(
        precipitation_hours=float(mean_values_glr_gaussian_model['mean_precipitation_hours']),
        sunshine_duration=float(mean_values_glr_gaussian_model['mean_sunshine_duration']),
        wind_speed=float(mean_values_glr_gaussian_model['mean_wind_speed'])
    )
])

mean_input_glr_gaussian_assembled_df = assembler.transform(mean_input_glr_gaussian_df)
mean_input_glr_gaussian_scaled_df = scaler_model.transform(mean_input_glr_gaussian_assembled_df)

mean_input_glr_gaussian_prediction_df = glr_gaussian_model.transform(mean_input_glr_gaussian_scaled_df)

mean_input_glr_gaussian_prediction_value = mean_input_glr_gaussian_prediction_df.collect()[0]['prediction']

print(f"Predicted ET₀ (Gaussian GLR): {mean_input_glr_gaussian_prediction_value:.3f} mm")

if mean_input_glr_gaussian_prediction_value < 1.5:
    print("✅ Mean conditions satisfy ET₀ < 1.5 mm")
else:
    print("❌ Mean conditions do NOT satisfy ET₀ < 1.5 mm")


In [ ]:
%spark.pyspark

mean_input_glr_gamma_df = spark.createDataFrame([
    Row(
        precipitation_hours=float(mean_values_glr_gamma_model['mean_precipitation_hours']),
        sunshine_duration=float(mean_values_glr_gamma_model['mean_sunshine_duration']),
        wind_speed=float(mean_values_glr_gamma_model['mean_wind_speed'])
    )
])

mean_input_glr_gamma_assembled_df = assembler.transform(mean_input_glr_gamma_df)
mean_input_glr_gamma_scaled_df = scaler_model.transform(mean_input_glr_gamma_assembled_df)

mean_input_glr_gamma_prediction_df = glr_gamma_model.transform(mean_input_glr_gamma_scaled_df)

mean_input_glr_gamma_prediction_value = mean_input_glr_gamma_prediction_df.collect()[0]['prediction']

print(f"Predicted ET₀ (Gamma GLR): {mean_input_glr_gamma_prediction_value:.3f} mm")

if mean_input_glr_gamma_prediction_value < 1.5:
    print("✅ Mean conditions satisfy ET₀ < 1.5 mm")
else:
    print("❌ Mean conditions do NOT satisfy ET₀ < 1.5 mm")


In [ ]:
%spark.pyspark
